<a href="https://colab.research.google.com/github/Mario-Cattaneo/SemesterProject/blob/main/Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Imports and Device Setup
import copy
import math
from dataclasses import dataclass, field
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Executing on device: {device}")

Executing on device: cpu


In [2]:
# Cell 2: Hierarchical Configuration with Unified Time Units (ms & bits)
from dataclasses import dataclass, field
import math
from typing import Dict, List, Tuple
import numpy as np
import torch
import torch.nn as nn


@dataclass
class ChannelConfig:
    capacity_bits_per_ms: (
        float  # Capacity in bits/ms (e.g., 30 Mbps = 30,000 bits/ms)
    )
    prop_delay_ms: float  # Propagation delay in ms
    bit_width: int = 8  # b bits per scalar component

    @classmethod
    def from_mbps(cls, capacity_mbps: float, prop_delay_ms: float, bit_width=8):
        """Helper: converts Mbps to bits/ms (1 Mbps = 1,000 bits/ms)."""
        return cls(
            capacity_bits_per_ms=capacity_mbps * 1000.0,
            prop_delay_ms=prop_delay_ms,
            bit_width=bit_width,
        )


@dataclass
class NodeConfig:
    device_name: str
    exec_time_ms: float  # Device execution delay t_i^exec in ms


@dataclass
class ToleranceConfig:
    epsilon_ms: float = 0.8  # Global tolerance band [t_target - eps, t_target]
    epsilon_margin_ms: float = (
        0.3  # Max allowed SLA ceiling margin (t_target + margin)
    )


@dataclass
class ALMConfig:
    t_max: float  # Target SLA in ms
    lambda1_init: float = 0.0
    lambda2_init: float = 0.005
    rho: float = 1.5
    tau_viol: float = 0.25
    e_step: int = 2


@dataclass
class GateConfig:
    alpha: float = 10.0  # Smooth-max sharpness (constant)
    beta_init: float = 1.5
    beta_min: float = 0.05
    decay_rate: float = 0.90
    gamma: float = -0.1
    zeta: float = 1.1
    s_init: float = 1.0


@dataclass
class ExperimentConfig:
    experiment_name: str = "TBN_Component_Level"
    batch_size: int = 128
    epochs: int = 8
    lr_s: float = 0.05  # Learning rate for component gates
    lr_c: float = 0.01  # Learning rate for continuous imputations
    nodes: Dict[int, NodeConfig] = field(default_factory=dict)
    edges: Dict[Tuple[int, int], ChannelConfig] = field(default_factory=dict)
    alm: ALMConfig = field(default_factory=lambda: ALMConfig(t_max=25.0))
    gate: GateConfig = field(default_factory=GateConfig)
    tolerance: ToleranceConfig = field(default_factory=ToleranceConfig)


print("Cell 2: Unified configuration layer ready.")

Cell 2: Unified configuration layer ready.


In [3]:
# Cell 3: True Component-Level (Element-Wise) Gating and Imputation


class EdgeBottleneck(nn.Module):
    """Component-level gating: learns one gate s and one imputation c for every

    individual scalar feature k in D_{i,j}.
    """

    def __init__(
        self,
        feature_shape: Tuple[int, ...],
        channel_cfg: ChannelConfig,
        gate_cfg: GateConfig,
    ):
        super().__init__()
        self.feature_shape = feature_shape  # e.g., (32, 32, 32)
        self.channel_cfg = channel_cfg
        self.gate_cfg = gate_cfg

        self.total_components = int(np.prod(feature_shape))

        # s and c match the EXACT tensor dimensions (C, H, W)
        self.s = nn.Parameter(torch.full(feature_shape, gate_cfg.s_init))
        self.c = nn.Parameter(torch.zeros(feature_shape))

    def get_transmission_prob(self, beta: float) -> torch.Tensor:
        """p(m_i,j^(k) > 0) evaluated component-wise."""
        offset = beta * math.log(-self.gate_cfg.gamma / self.gate_cfg.zeta)
        return torch.sigmoid(self.s - offset)

    def expected_transmission_time_ms(self, beta: float) -> torch.Tensor:
        """t_trans = (b * E[|Omega|]) / C [bits / (bits/ms) = ms]."""
        probs = self.get_transmission_prob(beta)
        expected_active_components = torch.sum(probs)
        total_bits = expected_active_components * self.channel_cfg.bit_width
        # Direct unified division: bits / (bits/ms) = ms
        trans_time_ms = total_bits / self.channel_cfg.capacity_bits_per_ms
        return trans_time_ms

    def forward(
        self, z: torch.Tensor, beta: float, deterministic: bool = False
    ) -> Tuple[torch.Tensor, float]:
        s_broad = self.s.unsqueeze(0)
        c_broad = self.c.unsqueeze(0)

        if not deterministic:
            eps = torch.rand_like(s_broad).clamp(1e-6, 1.0 - 1e-6)
            logits_eps = torch.log(eps) - torch.log(1.0 - eps)
            m_tilde = torch.sigmoid((logits_eps + s_broad) / beta)
            m_bar = (
                m_tilde * (self.gate_cfg.zeta - self.gate_cfg.gamma)
                + self.gate_cfg.gamma
            )
            m = torch.clamp(m_bar, 0.0, 1.0)

            # u^(k) = z^(k) * m^(k) + (1 - m^(k)) * c^(k)
            u = z * m + (1.0 - m) * c_broad
            active_fraction = float(m.mean().item())
            return u, active_fraction
        else:
            threshold = beta * math.log(
                -self.gate_cfg.gamma / self.gate_cfg.zeta
            )
            hard_mask = (s_broad >= threshold).float()

            u = z * hard_mask + (1.0 - hard_mask) * c_broad
            active_fraction = float(hard_mask.mean().item())
            return u, active_fraction


print("Cell 3: Component-level EdgeBottleneck compiled.")

Cell 3: Component-level EdgeBottleneck compiled.


In [4]:
# Cell 4: DAGTimingEngine with Direct Millisecond Summation


class DAGTimingEngine:

    def __init__(
        self,
        nodes: Dict[int, NodeConfig],
        edges: Dict[Tuple[int, int], ChannelConfig],
    ):
        self.nodes = nodes
        self.edges = edges

        self.parents: Dict[int, List[int]] = {i: [] for i in nodes}
        self.children: Dict[int, List[int]] = {i: [] for i in nodes}
        for u, v in edges.keys():
            self.parents[v].append(u)
            self.children[u].append(v)

        self.topo_order = self._get_topological_sort()
        self.terminal_node = self.topo_order[-1]

    def _get_topological_sort(self) -> List[int]:
        in_deg = {i: len(self.parents[i]) for i in self.nodes}
        queue = [i for i in self.nodes if in_deg[i] == 0]
        order = []
        while queue:
            curr = queue.pop(0)
            order.append(curr)
            for nxt in self.children[curr]:
                in_deg[nxt] -= 1
                if in_deg[nxt] == 0:
                    queue.append(nxt)
        return order

    def compute_relaxed_latency(
        self,
        bottlenecks: Dict[Tuple[int, int], EdgeBottleneck],
        beta: float,
        alpha: float,
    ) -> torch.Tensor:
        t_tilde: Dict[int, torch.Tensor] = {}

        for j in self.topo_order:
            t_exec = torch.tensor(
                self.nodes[j].exec_time_ms, device=bottlenecks[(0, 1)].s.device
            )

            if len(self.parents[j]) == 0:
                t_tilde[j] = t_exec
            else:
                branch_arrivals = []
                for i in self.parents[j]:
                    t_trans = bottlenecks[(i, j)].expected_transmission_time_ms(
                        beta
                    )
                    t_prop = torch.tensor(
                        self.edges[(i, j)].prop_delay_ms, device=t_trans.device
                    )
                    z_ij = (
                        t_tilde[i] + t_prop + t_trans
                    )  # All terms strictly in ms
                    branch_arrivals.append(z_ij)

                stacked = torch.stack(branch_arrivals)
                z_max = torch.max(stacked).detach()
                lse = z_max + (1.0 / alpha) * torch.log(
                    torch.sum(torch.exp(alpha * (stacked - z_max)))
                )
                t_tilde[j] = t_exec + lse

        return t_tilde[self.terminal_node]

    def compute_physical_latency(
        self,
        bottlenecks: Dict[Tuple[int, int], EdgeBottleneck],
        beta: float,
        deterministic: bool = True,
    ) -> float:
        t_physical: Dict[int, float] = {}

        for j in self.topo_order:
            t_exec = self.nodes[j].exec_time_ms

            if len(self.parents[j]) == 0:
                t_physical[j] = t_exec
            else:
                branch_delays = []
                for i in self.parents[j]:
                    bneck = bottlenecks[(i, j)]
                    if deterministic:
                        thresh = beta * math.log(
                            -bneck.gate_cfg.gamma / bneck.gate_cfg.zeta
                        )
                        active_components = (bneck.s >= thresh).sum().item()
                    else:
                        active_components = (
                            bneck.get_transmission_prob(beta).sum().item()
                        )

                    total_bits = (
                        active_components * bneck.channel_cfg.bit_width
                    )
                    # bits / (bits/ms) = ms
                    t_trans = (
                        total_bits / self.edges[(i, j)].capacity_bits_per_ms
                    )
                    t_prop = self.edges[(i, j)].prop_delay_ms
                    branch_delays.append(t_physical[i] + t_prop + t_trans)

                t_physical[j] = t_exec + max(branch_delays)

        return t_physical[self.terminal_node]

    def compute_irreducible_latency(self) -> float:
        """Physical floor when t_trans -> 0 (ms)."""
        t_floor: Dict[int, float] = {}
        for j in self.topo_order:
            t_exec = self.nodes[j].exec_time_ms
            if len(self.parents[j]) == 0:
                t_floor[j] = t_exec
            else:
                branch_delays = [
                    t_floor[i] + self.edges[(i, j)].prop_delay_ms
                    for i in self.parents[j]
                ]
                t_floor[j] = t_exec + max(branch_delays)
        return t_floor[self.terminal_node]


print("Cell 4: DAGTimingEngine ready.")

Cell 4: DAGTimingEngine ready.


In [5]:
# Cell 5: Collaborative ResNet DAG with Residual Micro-Splits


class ResBlock(nn.Module):

    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(
                channels,
                channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(channels),
            nn.ReLU(),
            nn.Conv2d(
                channels,
                channels,
                kernel_size=3,
                stride=1,
                padding=1,
                bias=False,
            ),
            nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        return F.relu(x + self.conv(x))


class CollaborativeResNetDAG(nn.Module):

    def __init__(self, cfg: ExperimentConfig):
        super().__init__()
        self.cfg = cfg

        # Device 0: Input Stem + ResBlock (Outputs 64 channels)
        self.node0_split = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            ResBlock(64),
        )

        # Device 1: Parallel Worker A (Processes 32 channels -> outputs 64)
        self.node1_split = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            ResBlock(64),
        )

        # Device 2: Parallel Worker B (Processes 32 channels -> outputs 64)
        self.node2_split = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            ResBlock(64),
        )

        # Device 3: Concatenation (64+64 = 128 channels) + ResBlock + Classifier
        self.node3_split = nn.Sequential(
            ResBlock(128),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(128, 10),
        )

        # EdgeBottlenecks for directed channels
        self.bottlenecks = nn.ModuleDict()
        self.bottlenecks["0->1"] = EdgeBottleneck(
            (32, 32, 32), cfg.edges[(0, 1)], cfg.gate
        )
        self.bottlenecks["0->2"] = EdgeBottleneck(
            (32, 32, 32), cfg.edges[(0, 2)], cfg.gate
        )
        self.bottlenecks["1->3"] = EdgeBottleneck(
            (64, 16, 16), cfg.edges[(1, 3)], cfg.gate
        )
        self.bottlenecks["2->3"] = EdgeBottleneck(
            (64, 16, 16), cfg.edges[(2, 3)], cfg.gate
        )

    def get_bottleneck_map(self) -> Dict[Tuple[int, int], EdgeBottleneck]:
        return {
            (0, 1): self.bottlenecks["0->1"],
            (0, 2): self.bottlenecks["0->2"],
            (1, 3): self.bottlenecks["1->3"],
            (2, 3): self.bottlenecks["2->3"],
        }

    def forward(
        self, x: torch.Tensor, beta: float, deterministic: bool = False
    ) -> torch.Tensor:
        z0 = self.node0_split(x)
        z0_A, z0_B = torch.split(z0, 32, dim=1)

        u01, _ = self.bottlenecks["0->1"](
            z0_A, beta, deterministic=deterministic
        )
        u02, _ = self.bottlenecks["0->2"](
            z0_B, beta, deterministic=deterministic
        )

        z1 = self.node1_split(u01)
        z2 = self.node2_split(u02)

        u13, _ = self.bottlenecks["1->3"](
            z1, beta, deterministic=deterministic
        )
        u23, _ = self.bottlenecks["2->3"](
            z2, beta, deterministic=deterministic
        )

        z_concat = torch.cat([u13, u23], dim=1)
        return self.node3_split(z_concat)


print("Cell 5: Upgraded residual architecture ready.")

Cell 5: Upgraded residual architecture ready.


In [6]:
# Cell 6: ALMTrainer & RatePenaltyTrainer with Tolerance Stopping and Max Iteration Guards
import copy
import torch.optim as optim


class ALMTrainer:

    def __init__(
        self,
        model: CollaborativeResNetDAG,
        timing_engine: DAGTimingEngine,
        cfg: ExperimentConfig,
    ):
        self.model = model.to(device)
        self.timing_engine = timing_engine
        self.cfg = cfg

        self.lambda1 = cfg.alm.lambda1_init
        self.lambda2 = cfg.alm.lambda2_init
        self.prev_violation = float("inf")

        s_params = [
            bneck.s for bneck in self.model.get_bottleneck_map().values()
        ]
        c_params = [
            bneck.c for bneck in self.model.get_bottleneck_map().values()
        ]
        self.optimizer = optim.Adam(
            [
                {"params": s_params, "lr": cfg.lr_s},
                {"params": c_params, "lr": cfg.lr_c},
            ]
        )
        self.criterion = nn.CrossEntropyLoss()

    def train_epoch(self, dataloader, epoch: int):
        self.model.train()
        beta = max(
            self.cfg.gate.beta_min, self.cfg.gate.beta_init * (self.cfg.gate.decay_rate**epoch)
        )
        bneck_map = self.model.get_bottleneck_map()

        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            self.optimizer.zero_grad()

            y_hat = self.model(x, beta=beta, deterministic=False)
            task_loss = self.criterion(y_hat, y)

            tilde_t = self.timing_engine.compute_relaxed_latency(
                bneck_map, beta=beta, alpha=self.cfg.gate.alpha
            )
            delta = torch.clamp(tilde_t - self.cfg.alm.t_max, min=0.0)
            latency_penalty = self.lambda1 * delta + self.lambda2 * (delta**2)

            loss = task_loss + latency_penalty
            loss.backward()
            self.optimizer.step()

        # Dual outer step
        if (epoch + 1) % self.cfg.alm.e_step == 0:
            with torch.no_grad():
                current_tilde_t = (
                    self.timing_engine.compute_relaxed_latency(
                        bneck_map, beta=beta, alpha=self.cfg.gate.alpha
                    ).item()
                )
                curr_viol = max(0.0, current_tilde_t - self.cfg.alm.t_max)
                self.lambda1 = max(
                    0.0, self.lambda1 + 2.0 * self.lambda2 * curr_viol
                )
                if curr_viol > self.cfg.alm.tau_viol * self.prev_violation:
                    self.lambda2 *= self.cfg.alm.rho
                self.prev_violation = curr_viol

    @torch.no_grad()
    def evaluate(self, dataloader) -> Tuple[float, float, Dict[str, float]]:
        self.model.eval()
        correct, total = 0, 0
        beta_eval = self.cfg.gate.beta_min
        bneck_map = self.model.get_bottleneck_map()

        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            y_hat = self.model(x, beta=beta_eval, deterministic=True)
            correct += (y_hat.argmax(dim=1) == y).sum().item()
            total += x.size(0)

        acc = 100.0 * correct / total
        lat = self.timing_engine.compute_physical_latency(
            bneck_map, beta=beta_eval, deterministic=True
        )

        retained = {}
        for edge_key, bneck in bneck_map.items():
            thresh = beta_eval * math.log(
                -bneck.gate_cfg.gamma / bneck.gate_cfg.zeta
            )
            retained[f"{edge_key[0]}->{edge_key[1]}"] = (
                (bneck.s >= thresh).float().mean().item() * 100.0
            )

        return acc, lat, retained

    def train_with_dual_termination(
        self, dataloader, test_loader, t_target: float, max_epochs: int = 10
    ):
        """Dual-termination: stops upon reaching the tolerance window [target -

        eps, target + margin] or upon hitting max_epochs.
        """
        eps = self.cfg.tolerance.epsilon_ms
        margin = self.cfg.tolerance.epsilon_margin_ms

        best_candidate_state = None
        best_acc = 0.0
        best_lat = float("inf")

        for ep in range(max_epochs):
            self.train_epoch(dataloader, ep)
            acc, lat, ret = self.evaluate(test_loader)

            # Primary Termination Condition: Tolerance Band Convergence
            if (t_target - eps) <= lat <= (t_target + margin):
                print(
                    f"  [TBN Converged] Epoch {ep+1}/{max_epochs} | Latency: {lat:.2f}ms in [{t_target-eps:.1f}, {t_target+margin:.1f}]ms | Acc: {acc:.2f}%"
                )
                return lat, acc, ret

            # Fallback tracker: best model that does not violate the SLA
            if lat <= (t_target + margin) and acc > best_acc:
                best_acc = acc
                best_lat = lat
                best_candidate_state = copy.deepcopy(self.model.state_dict())

        # Alternate Termination: Max epochs budget exhausted
        print(
            f"  [TBN Max Epochs Reached] Returning best checkpoint within SLA budget."
        )
        if best_candidate_state is not None:
            self.model.load_state_dict(best_candidate_state)
            acc, lat, ret = self.evaluate(test_loader)
            return lat, acc, ret
        return lat, acc, ret


class RatePenaltyTrainer:
    """Trains baselines using additive rate penalties with unified time units."""

    def __init__(
        self,
        model: CollaborativeResNetDAG,
        timing_engine: DAGTimingEngine,
        cfg: ExperimentConfig,
        weight: float,
        mode: str = "capacity_delay",
    ):
        self.model = model.to(device)
        self.timing_engine = timing_engine
        self.cfg = cfg
        self.weight = weight
        self.mode = mode

        s_params = [
            bneck.s for bneck in self.model.get_bottleneck_map().values()
        ]
        c_params = [
            bneck.c for bneck in self.model.get_bottleneck_map().values()
        ]
        self.optimizer = optim.Adam(
            [
                {"params": s_params, "lr": cfg.lr_s},
                {"params": c_params, "lr": cfg.lr_c},
            ]
        )
        self.criterion = nn.CrossEntropyLoss()

    def train_epoch(self, dataloader, epoch: int):
        self.model.train()
        beta = max(
            self.cfg.gate.beta_min, self.cfg.gate.beta_init * (self.cfg.gate.decay_rate**epoch)
        )
        bneck_map = self.model.get_bottleneck_map()

        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            self.optimizer.zero_grad()

            y_hat = self.model(x, beta=beta, deterministic=False)
            task_loss = self.criterion(y_hat, y)

            if self.mode == "uniform_bits":
                # Approach 1: weight * Sum(Bits / 1000) [kilobits]
                total_kbits = sum(
                    (
                        torch.sum(bneck.get_transmission_prob(beta))
                        * bneck.channel_cfg.bit_width
                    )
                    / 1000.0
                    for bneck in bneck_map.values()
                )
                rate_penalty = self.weight * total_kbits
            else:
                # Approach 2: weight * Sum(t_trans in ms)
                total_delay_ms = sum(
                    bneck.expected_transmission_time_ms(beta)
                    for bneck in bneck_map.values()
                )
                rate_penalty = self.weight * total_delay_ms

            loss = task_loss + rate_penalty
            loss.backward()
            self.optimizer.step()

    evaluate = ALMTrainer.evaluate


print("Cell 6: Tolerance-aware trainers compiled.")

Cell 6: Tolerance-aware trainers compiled.


In [10]:
# ==============================================================================
# Cell 6B: Universal Backbone Checkpoint Manager & Fast Local Dataset
# ==============================================================================
import copy
import os
import shutil
from google.colab import drive
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# 1. Mount Google Drive
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# 2. Fast Local Disk Copy (Eliminates Google Drive Network I/O Latency & Deadlocks)
drive_cifar_path = "/content/drive/MyDrive/Colab_Datasets/CIFAR10"
local_cifar_path = "/content/cifar10_local"

if not os.path.exists(local_cifar_path):
    print(
        "Copying CIFAR-10 from Google Drive to local fast disk (takes ~3s)..."
    )
    shutil.copytree(drive_cifar_path, local_cifar_path)
    print("Done! Dataset is now cached locally on fast NVMe disk.")
else:
    print("Local fast CIFAR-10 cache already ready.")

# 3. DataLoaders (num_workers=0 avoids multiprocessing deadlocks in Colab)
transform_train = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
        ),
    ]
)
transform_test = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(
            (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
        ),
    ]
)

trainset = torchvision.datasets.CIFAR10(
    root=local_cifar_path, train=True, download=False, transform=transform_train
)
testset = torchvision.datasets.CIFAR10(
    root=local_cifar_path, train=False, download=False, transform=transform_test
)

train_loader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True, num_workers=0, pin_memory=True
)
test_loader = torch.utils.data.DataLoader(
    testset, batch_size=256, shuffle=False, num_workers=0, pin_memory=True
)

print(f"DataLoaders ready: {len(trainset)} train, {len(testset)} test samples.")

# 4. Checkpoint Management (Persistent in Drive)
checkpoint_path = (
    "/content/drive/MyDrive/Colab_Datasets/universal_resnet_split.pt"
)

if os.path.exists(checkpoint_path):
    print(f"\nLoading universal pretrained weights from: {checkpoint_path}")
    pretrained_weights = torch.load(checkpoint_path, map_location=device)
    print(">>> Checkpoint loaded in 0.1s! Ready to run Cell 7 without waiting.")
else:
    print(
        "\nCheckpoint not found. Pretraining universal backbone on full CIFAR-10 (20 epochs)..."
    )
    dummy_cfg = ExperimentConfig(
        nodes={
            0: NodeConfig("N0", 1.0),
            1: NodeConfig("N1", 1.0),
            2: NodeConfig("N2", 1.0),
            3: NodeConfig("N3", 1.0),
        },
        edges={
            (0, 1): ChannelConfig.from_mbps(10.0, 1.0),
            (0, 2): ChannelConfig.from_mbps(10.0, 1.0),
            (1, 3): ChannelConfig.from_mbps(10.0, 1.0),
            (2, 3): ChannelConfig.from_mbps(10.0, 1.0),
        },
    )
    base_model = CollaborativeResNetDAG(dummy_cfg).to(device)

    for bneck in base_model.get_bottleneck_map().values():
        bneck.s.requires_grad = False
        bneck.c.requires_grad = False
        bneck.s.data.fill_(10.0)

    opt = torch.optim.AdamW(base_model.parameters(), lr=2e-3, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=20)
    crit = nn.CrossEntropyLoss()

    for ep in range(20):
        base_model.train()
        correct, total = 0, 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            y_hat = base_model(x, beta=1.0, deterministic=True)
            loss = crit(y_hat, y)
            loss.backward()
            opt.step()
            correct += (y_hat.argmax(dim=1) == y).sum().item()
            total += x.size(0)
        sched.step()
        if (ep + 1) % 5 == 0 or ep == 19:
            print(
                f"Pretrain Epoch {ep+1:02d}/20 | Acc: {100.0 * correct / total:.2f}%"
            )

    pretrained_weights = copy.deepcopy(base_model.state_dict())
    torch.save(pretrained_weights, checkpoint_path)
    print(f"\nSaved universal weights to: {checkpoint_path}")

Copying CIFAR-10 from Google Drive to local fast disk (takes ~3s)...
Done! Dataset is now cached locally on fast NVMe disk.
DataLoaders ready: 50000 train, 10000 test samples.

Loading universal pretrained weights from: /content/drive/MyDrive/Colab_Datasets/universal_resnet_split.pt
>>> Checkpoint loaded in 0.1s! Ready to run Cell 7 without waiting.


In [11]:
# ==============================================================================
# Cell 7: Fast, Verbose Benchmark with Local Disk & Dual Termination
# ==============================================================================
import copy
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

# 1. Fast Local DataLoader (Zero Deadlocks, num_workers=0)
transform_train = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
        ),
    ]
)
transform_test = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize(
            (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
        ),
    ]
)

local_path = (
    "/content/cifar10_local"
    if os.path.exists("/content/cifar10_local")
    else cifar_path
)

trainset = torchvision.datasets.CIFAR10(
    root=local_path, train=True, download=False, transform=transform_train
)
testset = torchvision.datasets.CIFAR10(
    root=local_path, train=False, download=False, transform=transform_test
)

train_loader = torch.utils.data.DataLoader(
    trainset, batch_size=128, shuffle=True, num_workers=0, pin_memory=True
)
test_loader = torch.utils.data.DataLoader(
    testset, batch_size=256, shuffle=False, num_workers=0, pin_memory=True
)

# 2. Hardware Topology (Config 2: Capacity Asymmetry)
nodes = {
    0: NodeConfig("Ingress_Pi", exec_time_ms=4.0),
    1: NodeConfig("Worker_A_Jetson", exec_time_ms=3.0),
    2: NodeConfig("Worker_B_Pi", exec_time_ms=7.0),
    3: NodeConfig("Terminal_Server", exec_time_ms=2.0),
}
edges = {
    (0, 1): ChannelConfig.from_mbps(capacity_mbps=30.0, prop_delay_ms=1.0),
    (0, 2): ChannelConfig.from_mbps(capacity_mbps=30.0, prop_delay_ms=1.0),
    (1, 3): ChannelConfig.from_mbps(
        capacity_mbps=50.0, prop_delay_ms=1.0
    ),  # Fast Link (50 Mbps)
    (2, 3): ChannelConfig.from_mbps(
        capacity_mbps=5.0, prop_delay_ms=1.0
    ),  # Bottleneck (5 Mbps)
}

timing_engine = DAGTimingEngine(nodes, edges)
eval_cfg = ExperimentConfig(
    nodes=nodes,
    edges=edges,
    lr_s=0.05,
    tolerance=ToleranceConfig(epsilon_ms=0.8, epsilon_margin_ms=0.3),
)

t_irreducible = timing_engine.compute_irreducible_latency()


def load_backbone_weights(model, weights_dict):
    filtered_state = {
        k: v
        for k, v in weights_dict.items()
        if not (k.endswith(".s") or k.endswith(".c"))
    }
    model.load_state_dict(filtered_state, strict=False)


def get_clean_model():
    m = CollaborativeResNetDAG(eval_cfg).to(device)
    load_backbone_weights(m, pretrained_weights)
    for name, p in m.named_parameters():
        if name.endswith(".s") or name.endswith(".c"):
            p.requires_grad = True
            if name.endswith(".s"):
                p.data.fill_(1.0)
            if name.endswith(".c"):
                p.data.zero_()
        else:
            p.requires_grad = False
    return m


# 3. Evaluate Uncompressed Baseline Anchor
uncomp_model = CollaborativeResNetDAG(eval_cfg).to(device)
load_backbone_weights(uncomp_model, pretrained_weights)
for bneck in uncomp_model.get_bottleneck_map().values():
    bneck.s.data.fill_(10.0)

base_acc, base_lat, _ = ALMTrainer.evaluate(
    ALMTrainer(uncomp_model, timing_engine, eval_cfg), test_loader
)

print("===================================================================")
print(
    f">>> [ANCHOR] Uncompressed Baseline: Latency = {base_lat:.2f} ms | Accuracy = {base_acc:.2f}%"
)
print(
    f">>> [ASYMPTOTE] Physical Irreducible Floor: Latency = {t_irreducible:.2f} ms"
)
print("===================================================================\n")

# 4. Evaluate Bottleneck Floor Anchor (With live progress logging)
print(
    "Evaluating Bottleneck Floor (Link 2->3 Transmit = 0%, Link 1->3 Active)..."
)
floor_model = get_clean_model()
bneck_map = floor_model.get_bottleneck_map()
bneck_map[(0, 2)].s.data.fill_(-10.0)
bneck_map[(0, 2)].s.requires_grad = False
bneck_map[(2, 3)].s.data.fill_(-10.0)
bneck_map[(2, 3)].s.requires_grad = False

floor_opt = torch.optim.Adam(
    [p for p in floor_model.parameters() if p.requires_grad], lr=0.01
)
crit = nn.CrossEntropyLoss()

for ep in range(3):  # 3 epochs is plenty
    floor_model.train()
    running_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        floor_opt.zero_grad()
        loss = crit(floor_model(x, beta=0.05, deterministic=True), y)
        loss.backward()
        floor_opt.step()
        running_loss += loss.item()
    print(
        f"  [Floor Epoch {ep+1}/3] Loss: {running_loss / len(train_loader):.4f}"
    )

floor_acc, floor_lat, _ = ALMTrainer.evaluate(
    ALMTrainer(floor_model, timing_engine, eval_cfg), test_loader
)
print(
    f">>> BOTTLENECK FLOOR ANCHOR: Latency = {floor_lat:.2f} ms | Accuracy = {floor_acc:.2f}%\n"
)

# 5. Partition Range into K Uniform SLA Targets
K = 3
t_span = base_lat - t_irreducible
t_targets = [round(base_lat - (k / (K + 1)) * t_span, 1) for k in range(1, K + 1)]
eps = eval_cfg.tolerance.epsilon_ms
margin = eval_cfg.tolerance.epsilon_margin_ms
print(f"Target SLA Checkpoints: {t_targets} ms (Tolerance band: +/- {eps} ms)\n")


# 6. Log-Secant Root Finder for Baselines with Dual Termination
def search_baseline_with_dual_termination(
    target_sla, mode, bracket_init, max_iter=3
):
    lam_low, lam_high = bracket_init
    t_low = base_lat
    t_high = t_irreducible + 0.5

    best_candidate_state = None
    best_acc = 0.0
    best_lat = float("inf")

    clean_init = get_clean_model()
    curr_state = copy.deepcopy(clean_init.state_dict())

    for it in range(max_iter):
        w = (t_low - target_sla) / max(1e-3, (t_low - t_high))
        w = float(np.clip(w, 0.05, 0.95))
        log_lam = (1.0 - w) * np.log(lam_low) + w * np.log(lam_high)
        lam_next = float(np.exp(log_lam))

        m = CollaborativeResNetDAG(eval_cfg).to(device)
        m.load_state_dict(curr_state)
        for name, p in m.named_parameters():
            p.requires_grad = (
                True if (name.endswith(".s") or name.endswith(".c")) else False
            )

        trainer = RatePenaltyTrainer(
            m, timing_engine, eval_cfg, weight=lam_next, mode=mode
        )
        for ep in range(2):  # 2 fast fine-tuning epochs per step
            trainer.train_epoch(train_loader, ep)

        acc, lat, _ = trainer.evaluate(test_loader)
        print(
            f"    [{mode} Step {it+1}/{max_iter}] weight={lam_next:.4f} -> Latency: {lat:.2f}ms (Target: {target_sla}ms) | Acc: {acc:.2f}%"
        )

        # Primary Exit: Inside tolerance band
        if (target_sla - eps) <= lat <= (target_sla + margin):
            print(
                f"      => Converged inside tolerance band [{target_sla-eps:.1f}, {target_sla+margin:.1f}]ms!"
            )
            return lat, acc

        # Record candidate
        if lat <= (target_sla + margin):
            if acc > best_acc or best_lat > (target_sla + margin):
                best_acc, best_lat = acc, lat
                best_candidate_state = copy.deepcopy(m.state_dict())
            lam_high, t_high = lam_next, lat
            curr_state = copy.deepcopy(m.state_dict())
        else:
            lam_low, t_low = lam_next, lat

    print(f"      => Returning closest candidate within SLA budget.")
    if best_candidate_state is not None:
        return best_lat, best_acc
    return lat, acc


# 7. Benchmark All 3 Approaches Across Target SLAs
tbn_results = []
app2_results = []
app1_results = []

for sla in t_targets:
    print(f"\n=======================================================")
    print(
        f"  BENCHMARKING TARGET SLA = {sla:.1f} ms (Band: [{sla-eps:.1f}, {sla+margin:.1f}] ms)"
    )
    print(f"=======================================================")

    # A. Proposed TBN (ALM with dual termination)
    print(f"--> Running TBN (Critical-Path ALM)...")
    cfg_tbn = ExperimentConfig(
        nodes=nodes, edges=edges, alm=ALMConfig(t_max=sla)
    )
    trainer_tbn = ALMTrainer(get_clean_model(), timing_engine, cfg_tbn)
    lat_tbn, acc_tbn, ret_tbn = trainer_tbn.train_with_dual_termination(
        train_loader, test_loader, t_target=sla, max_epochs=6
    )
    tbn_results.append((lat_tbn, acc_tbn))
    print(
        f"  [TBN Result] Latency: {lat_tbn:.2f} ms | Acc: {acc_tbn:.2f}% | Retained: {ret_tbn}\n"
    )

    # B. Approach 2: Capacity-Tied Delay Regularization (Log-Secant Search)
    print(f"--> Running Approach 2 (Capacity-Tied) Search...")
    lat2, acc2 = search_baseline_with_dual_termination(
        sla, mode="capacity_delay", bracket_init=(0.005, 2.5), max_iter=3
    )
    app2_results.append((lat2, acc2))
    print(f"  [App 2 Result] Latency: {lat2:.2f} ms | Acc: {acc2:.2f}%\n")

    # C. Approach 1: VDDIB Uniform Bit Penalty (Log-Secant Search)
    print(f"--> Running Approach 1 (VDDIB Uniform Bits) Search...")
    lat1, acc1 = search_baseline_with_dual_termination(
        sla, mode="uniform_bits", bracket_init=(0.05, 15.0), max_iter=3
    )
    app1_results.append((lat1, acc1))
    print(f"  [App 1 Result] Latency: {lat1:.2f} ms | Acc: {acc1:.2f}%\n")

# 8. Print Aligned Summary Table
print("\n" + "=" * 78)
print(
    f"{'Target SLA':<14} | {'TBN (Ours: ALM)':<18} | {'Capacity-Tied (App 2)':<20} | {'VDDIB (App 1)':<18}"
)
print("=" * 78)
print(
    f"{'Baseline (' + str(round(base_lat,1)) + 'ms)':<14} | {base_acc:.2f}%{'':<12} | {base_acc:.2f}%{'':<14} | {base_acc:.2f}%"
)

for i, sla in enumerate(t_targets):
    tbn_str = f"{tbn_results[i][1]:.2f}% ({tbn_results[i][0]:.1f}ms)"
    app2_str = f"{app2_results[i][1]:.2f}% ({app2_results[i][0]:.1f}ms)"
    app1_str = f"{app1_results[i][1]:.2f}% ({app1_results[i][0]:.1f}ms)"
    print(f"{str(sla) + ' ms':<14} | {tbn_str:<18} | {app2_str:<20} | {app1_str:<18}")

floor_str = f"{floor_acc:.2f}% ({floor_lat:.1f}ms)"
print(
    f"{'Floor (' + str(round(floor_lat,1)) + 'ms)':<14} | {floor_str:<18} | {floor_str:<20} | {floor_str:<18}"
)
print("=" * 78 + "\n")

# 9. Publication Plot: Aligned Trade-off Curves
plt.figure(figsize=(10, 6))

tbn_lats = [base_lat] + [pt[0] for pt in tbn_results] + [floor_lat]
tbn_accs = [base_acc] + [pt[1] for pt in tbn_results] + [floor_acc]
plt.plot(
    tbn_lats,
    tbn_accs,
    "o-",
    color="crimson",
    linewidth=2.4,
    markersize=7,
    label="TBN (Ours: Critical-Path ALM)",
)

app2_lats = [base_lat] + [pt[0] for pt in app2_results] + [floor_lat]
app2_accs = [base_acc] + [pt[1] for pt in app2_results] + [floor_acc]
plt.plot(
    app2_lats,
    app2_accs,
    "s--",
    color="royalblue",
    linewidth=1.8,
    markersize=6,
    label="Capacity-Tied (Delay Regularization)",
)

app1_lats = [base_lat] + [pt[0] for pt in app1_results] + [floor_lat]
app1_accs = [base_acc] + [pt[1] for pt in app1_results] + [floor_acc]
plt.plot(
    app1_lats,
    app1_accs,
    "^:",
    color="darkorange",
    linewidth=1.8,
    markersize=6,
    label="VDDIB-Style (Uniform Bit Penalty)",
)

plt.scatter(
    [base_lat],
    [base_acc],
    color="gold",
    edgecolors="black",
    s=190,
    zorder=6,
    marker="*",
    label="Uncompressed Baseline (100% Tx)",
)
plt.scatter(
    [floor_lat],
    [floor_acc],
    color="purple",
    edgecolors="black",
    s=140,
    zorder=6,
    marker="D",
    label=f"Bottleneck Floor ({floor_acc:.1f}%)",
)

for sla in t_targets:
    plt.axvspan(sla - eps, sla + margin, color="gray", alpha=0.15)
    plt.axvline(x=sla, color="gray", linestyle=":", alpha=0.6)

plt.axvline(
    x=t_irreducible,
    color="black",
    linestyle="--",
    alpha=0.8,
    label=f"Physical Floor ({t_irreducible:.1f}ms)",
)

plt.xlabel("Physical Inference Latency (ms)", fontsize=12)
plt.ylabel("CIFAR-10 Test Accuracy (%)", fontsize=12)
plt.title(
    "Aligned Pareto Evaluation with Global SLA Tolerance Bands", fontsize=13
)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend(loc="lower left", fontsize=10)
plt.tight_layout()
plt.show()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 